In [23]:
import os
import json
import pandas as pd
import tiktoken
from collections import defaultdict

# 初始化 GPT tokenizer
tokenizer = tiktoken.encoding_for_model("gpt-3.5-turbo")

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text))

def analyze_aspect_by_group(file_paths):
    group_stats = defaultdict(lambda: {
        "total_files": 0,
        "total_aspects": 0,
        "total_characters": 0,
        "total_tokens": 0,
        "total_doc_characters": 0,
        "total_doc_tokens": 0,
    })

    for path in file_paths:
        try:
            parts = path.split("/")
            aspect_number = int(parts[2])
        except (IndexError, ValueError):
            print(f"跳過無效路徑: {path}")
            continue

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        aspect_summary = data.get("aspect_summary", {})
        num_aspects = len(aspect_summary)
        document = data.get("document", "")
        doc_char_len = len(document)
        doc_token_len = count_tokens(document)

        group = group_stats[aspect_number]
        group["total_files"] += 1
        group["total_aspects"] += num_aspects
        group["total_doc_characters"] += doc_char_len
        group["total_doc_tokens"] += doc_token_len

        for summary in aspect_summary.values():
            group["total_characters"] += len(summary)
            group["total_tokens"] += count_tokens(summary)

    # 加總整體
    all_group = {
        "total_files": 0,
        "total_aspects": 0,
        "total_characters": 0,
        "total_tokens": 0,
        "total_doc_characters": 0,
        "total_doc_tokens": 0,
    }

    for group in group_stats.values():
        for key in all_group:
            all_group[key] += group[key]
    group_stats["all"] = all_group

    # 組成 DataFrame
    result_rows = []
    for aspect_number, stats in sorted(group_stats.items(), key=lambda x: (str(x[0]) != "all", x[0])):
        total_files = stats["total_files"]
        total_aspects = stats["total_aspects"]
        total_characters = stats["total_characters"]
        total_tokens = stats["total_tokens"]
        total_doc_characters = stats["total_doc_characters"]
        total_doc_tokens = stats["total_doc_tokens"]

        result_rows.append({
            "aspect_number": aspect_number,
            "total_files": total_files,
            "total_aspects": total_aspects,
            # "avg_characters_per_file": round((total_characters / total_files), 2) if total_files else 0, #四捨五入到第二位小數
            "avg_tokens_per_file": round((total_tokens / total_files),2) if total_files else 0,
            # "avg_characters_per_aspect": total_characters / total_aspects if total_aspects else 0,
            "avg_tokens_per_aspect": round((total_tokens / total_aspects), 2) if total_aspects else 0,
            # "avg_doc_characters": total_doc_characters / total_files if total_files else 0,
            "avg_doc_tokens": round((total_doc_tokens / total_files), 2) if total_files else 0,
        })

    return pd.DataFrame(result_rows)

# =========== Only test.json ================
# # 讀入 JSON 路徑清單
# with open("test.json", "r", encoding="utf-8") as f:
#     json_file_paths = json.load(f)

# # 執行分析
# df_grouped_stats = analyze_aspect_by_group(json_file_paths)

# # 👉 調整順序：確保 'all' 在最後一列
# df_grouped_stats["aspect_number_sort_key"] = df_grouped_stats["aspect_number"].apply(lambda x: float('inf') if x == "all" else int(x))
# df_grouped_stats = df_grouped_stats.sort_values("aspect_number_sort_key").drop(columns="aspect_number_sort_key")

# # 輸出 CSV
# df_grouped_stats.to_csv("aspect_document_grouped_summary_stats.csv", index=False)


# ✅ 讀入 train.json 和 test.json
with open("train.json", "r", encoding="utf-8") as f:
    train_paths = json.load(f)

with open("test.json", "r", encoding="utf-8") as f:
    test_paths = json.load(f)

# ✅ 合併兩份資料
combined_paths = train_paths + test_paths

# ✅ 執行分析（合併後的路徑）
df_grouped_stats = analyze_aspect_by_group(combined_paths)

# ✅ 確保 'all' 在最後一列
df_grouped_stats["aspect_number_sort_key"] = df_grouped_stats["aspect_number"].apply(
    lambda x: float('inf') if x == "all" else int(x)
)
df_grouped_stats = df_grouped_stats.sort_values("aspect_number_sort_key").drop(columns="aspect_number_sort_key")

# ✅ 輸出合併後統計結果
df_grouped_stats.to_csv("aspect_document_grouped_summary_stats.csv", index=False)

